In [72]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display,Markdown,update_display
from bs4 import BeautifulSoup
import requests
import gradio as gr




In [82]:
load_dotenv(override=True)
google_api_key=os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print(google_api_key[:5])
else:
    print("api key is not found")    


AQ.Ab


In [74]:
gemini_url="https://generativelanguage.googleapis.com/v1beta/openai/"
gemini=OpenAI(base_url=gemini_url,api_key=google_api_key)
ollama=OpenAI(base_url="http://127.0.0.1:11434/v1",api_key="ollama")


In [75]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


In [76]:
system_prompt = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [77]:
def stream_gemini(prompt):
    messages=[{"role":"system","content":system_prompt},{"role":"user","content":prompt}]
    
    stream=gemini.chat.completions.create(model="gemini-3.1-flash-lite",messages=messages,stream=True)
    result=""
    for chunks in stream:
        result+=chunks.choices[0].delta.content or ""
    yield result


In [78]:
def stream_ollama(prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
      ]
    stream = ollama.chat.completions.create(
        model='llama3.2',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [79]:
def stream_brochure(name,url,model):
    
    prompt=f"this is the company name:{name},and this is the bronchure of this company , the landing page is \n  "
    prompt+=fetch_website_contents(url)
    if model=="GEMINI":
        stream=stream_gemini(prompt)
    elif model=="OLLAMA":
        stream=stream_ollama(prompt) 
    else:
        raise ValueError("unknown model")
    yield from stream    
    


In [80]:
for text in stream_brochure("google","https://google.com","GEMINI"):
    print(text)

# Google: Organizing the World’s Information

### Empowering Global Connectivity and Innovation
At Google, our mission is to organize the world’s information and make it universally accessible and useful. We are a global technology leader dedicated to developing transformative tools that bridge the gap between human curiosity and digital solutions.

### For Our Customers: Seamless Integration
We provide the ecosystem that powers your digital life. From the world’s most intuitive search engine to productivity suites like Gmail, our platforms are designed to help you discover, create, and communicate effortlessly. Whether you are searching for answers or managing your daily workflow, Google is the gateway to your digital world.

### For Our Investors: Driving the Future of AI
Google is at the forefront of the artificial intelligence revolution. With cutting-edge investments in machine learning, generative AI, and data processing, we are building the infrastructure of the future. Our comm

In [83]:
name_input=gr.Textbox(label="company name: ")
url_input=gr.Textbox(label="company url: ",placeholder="https://example.com")
model_selector=gr.Dropdown(choices=["GEMINI","OLLAMA"],label="select model",value="GEMINI")
message_output=gr.Markdown(label="response from AI: ")

view=gr.Interface(fn=stream_brochure,
    title="SiteBrochure AI",
    inputs=[name_input,url_input,model_selector],
    outputs=[message_output],
    flagging_mode="never")
    
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.
